# 05 · Profile: graduates (IPEDS completions + directory, CUNY Programs)

**Method:** `docs/framework.md` §3 (the 10-section exploration template). Cards: `docs/source-cards/ipeds_completions.md`, `docs/source-cards/cuny_programs.md`.

**Why these sources matter.** brief Q3 asks *"how many people graduate in each major, every year?"*. That's the supply side of the gap.
- **IPEDS Completions** (federal, NCES) counts every degree and certificate awarded by every college in NYC, by major (CIP code) and award level. Together with the IPEDS directory (HD), it tells us which colleges are in NYC and which are CUNY. That's the "all NYC colleges, CUNY highlighted" scope (brief S3).
- **CUNY Programs** (data.ny.gov `28gk-bu58`) is CUNY's own program-level file: graduates and fall enrollment per program, one year more recent than IPEDS (to AY 2024-25 / Fall 2025).
- Two small CUNY files are used only for tie-out: degrees by type (`ybg5-afvs`) and enrollment by college (`487n-fp4r`).

**What this notebook does:** reads the raw files, checks grain, codes, coverage and arithmetic, ties CUNY's numbers to IPEDS, and ends with an issues list and proposed staging tables. It changes nothing.

Every read uses `dtype=str, keep_default_na=False`. Reading ten years of IPEDS takes about a minute.

In [1]:
import io
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NYC_COUNTIES = {"36005": "Bronx", "36047": "Brooklyn (Kings)", "36061": "Manhattan (New York)",
                "36081": "Queens", "36085": "Staten Island (Richmond)"}
KEY = ["UNITID", "CIPCODE", "MAJORNUM", "AWLEVEL"]
YEARS = range(2015, 2025)  # IPEDS file year = the year the academic year ends (C2024 = July 2023-June 2024)

## 0. Source header

| | **IPEDS Completions** (`ipeds_completions`) + Directory (`ipeds_directory`) | **CUNY Programs** (`cuny_programs`) |
|---|---|---|
| **Publisher** | U.S. Dept. of Education, NCES, Integrated Postsecondary Education Data System | CUNY Office of Institutional Research, via data.ny.gov `28gk-bu58` |
| **Dataset** | `C<YYYY>_A.zip` (awards by college × major × level) and `HD<YYYY>.zip` (one row per college), 2015–2024 | *CUNY Graduates and Enrollment by Program* |
| **How it was collected** | **Administrative census:** every college that takes federal student aid must report (Title IV), so coverage is near-complete. Counts are awards, not people. Each award is reported under up to two majors (`MAJORNUM` 1 and 2) | **Administrative records** from CUNY's student system, by registered program (NYSED program code = `IRP CODE`) |
| **Reference period** | Award year July 1 → June 30. File `C2024` = awards from July 2023 to June 2024 | Academic years (`AY_2007_2008` …) and fall terms (`Fall_2008` …) |
| **Revisions** | Each year is published provisional first, then revised a year later. The zip then holds both (`…_rv.csv`) | One current vintage; the portal overwrites it |
| **License** | U.S. government work, public domain | NY Open Data |

Documentation: [IPEDS Completions survey](https://nces.ed.gov/ipeds/survey-components/7), [data files](https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx), dictionary `C2024_A_Dict.zip` (in `data/raw`), [CUNY dataset page](https://data.ny.gov/d/28gk-bu58).

In [2]:
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILES = man[man.key.str.startswith(("ipeds", "cuny", "nces_cip"))][["key", "file", "downloaded_utc", "note"]]
FILES.groupby("key").agg(files=("file", "size"), first=("file", "first"), last=("file", "last"))

,files,first,last
key,,,
cuny_degrees_by_type,1,graduates/cuny_degrees_by_type/ybg5-afvs.csv,graduates/cuny_degrees_by_type/ybg5-afvs.csv
cuny_enrollment,1,graduates/cuny_enrollment/487n-fp4r.csv,graduates/cuny_enrollment/487n-fp4r.csv
cuny_programs,1,graduates/cuny_programs/28gk-bu58.csv,graduates/cuny_programs/28gk-bu58.csv
ipeds_completions,10,graduates/ipeds_completions/C2015_A.zip,graduates/ipeds_completions/C2024_A.zip
ipeds_dictionary,1,graduates/ipeds_dictionary/C2024_A_Dict.zip,graduates/ipeds_dictionary/C2024_A_Dict.zip
ipeds_directory,10,graduates/ipeds_directory/HD2015.zip,graduates/ipeds_directory/HD2024.zip
nces_cip,3,graduates/nces_cip/Crosswalk2010to2020.csv,graduates/nces_cip/CIP2020_SOC2018_Crosswalk.xlsx


## 1. Load as text

**IPEDS zips aren't uniform.** Look inside each one before reading: which CSVs it holds, and what text encoding it uses.

In [3]:
def decode(raw):
    # IPEDS switched from Latin-1 to UTF-8 partway through; try UTF-8 first
    try:
        return raw.decode("utf-8-sig"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("latin-1"), "latin-1"

def read_member(zpath, member, **kw):
    text, enc = decode(zipfile.ZipFile(zpath).read(member))
    return pd.read_csv(io.StringIO(text), dtype=str, keep_default_na=False, **kw), enc

inv = []
for y in YEARS:
    for kind, folder in [("C", "ipeds_completions"), ("HD", "ipeds_directory")]:
        zp = RAW / "graduates" / folder / (f"C{y}_A.zip" if kind == "C" else f"HD{y}.zip")
        members = [m for m in zipfile.ZipFile(zp).namelist() if m.lower().endswith(".csv")]
        _, enc = decode(zipfile.ZipFile(zp).read(members[0]))
        inv.append({"year": y, "file": kind, "csv members": ", ".join(members), "encoding": enc})
pd.DataFrame(inv).pivot(index="year", columns="file", values=["csv members", "encoding"])

csv members             encoding         
file                            C          HD        C       HD
year                                                           
2015  c2015_a.csv, c2015_a_rv.csv  hd2015.csv    utf-8  latin-1
2016  c2016_a.csv, c2016_a_rv.csv  hd2016.csv    utf-8  latin-1
2017  c2017_a.csv, c2017_a_rv.csv  hd2017.csv    utf-8  latin-1
2018  c2018_a.csv, c2018_a_rv.csv  hd2018.csv    utf-8  latin-1
2019  c2019_a.csv, c2019_a_rv.csv  hd2019.csv    utf-8  latin-1
2020  c2020_a.csv, c2020_a_rv.csv  hd2020.csv    utf-8  latin-1
2021  c2021_a.csv, c2021_a_rv.csv  hd2021.csv    utf-8  latin-1
2022  c2022_a.csv, c2022_a_rv.csv  hd2022.csv    utf-8  latin-1
2023  C2023_a.csv, C2023_a_RV.csv  HD2023.csv    utf-8    utf-8
2024                  C2024_a.csv  HD2024.csv    utf-8    utf-8

**What the revised file changes.** Where a zip holds both versions, compare them on the key:

In [4]:
usecols = KEY + ["XCTOTALT", "CTOTALT", "CTOTALM", "CTOTALW"]
comp, rv_diff = {}, []
for y in YEARS:
    zp = RAW / "graduates" / "ipeds_completions" / f"C{y}_A.zip"
    members = [m for m in zipfile.ZipFile(zp).namelist() if m.lower().endswith(".csv")]
    rv = [m for m in members if "_rv" in m.lower()]
    orig, _ = read_member(zp, next(m for m in members if m not in rv), usecols=usecols)
    if rv:
        rev, _ = read_member(zp, rv[0], usecols=usecols)
        m = orig.merge(rev, on=KEY, how="outer", suffixes=("_orig", "_rv"), indicator=True)
        rv_diff.append({"year": y, "rows only in original": (m._merge == "left_only").sum(),
                        "rows only in revised": (m._merge == "right_only").sum(),
                        "counts changed": ((m._merge == "both") & (m.CTOTALT_orig != m.CTOTALT_rv)).sum(),
                        "US awards, original": pd.to_numeric(orig.CTOTALT).sum(),
                        "US awards, revised": pd.to_numeric(rev.CTOTALT).sum()})
        comp[y] = rev.assign(release="revised")
    else:
        comp[y] = orig.assign(release="provisional")
    comp[y].insert(0, "year", y)
d = pd.DataFrame(rv_diff).set_index("year")
d["change"] = d["US awards, revised"] / d["US awards, original"] - 1
display(d.style.format({"US awards, original": "{:,}", "US awards, revised": "{:,}", "change": "{:+.3%}"}))

dict_intro = pd.read_excel(zipfile.ZipFile(RAW / "graduates/ipeds_dictionary/C2024_A_Dict.zip").open("C2024_a_dict.xlsx"),
                           sheet_name="Introduction", header=None, dtype=str).fillna("")
print("C2024 dictionary says:", " | ".join(v for v in dict_intro.iloc[:3, 0] if v))

,rows only in original,rows only in revised,counts changed,"US awards, original","US awards, revised",change
year,,,,,,
2015,38,79,392,"10,107,616","10,109,316",+0.017%
2016,51,110,543,"10,167,408","10,168,310",+0.009%
2017,246,185,516,"10,280,596","10,280,230",-0.004%
2018,61,91,489,"10,456,508","10,458,220",+0.016%
2019,140,60,354,"10,637,492","10,641,080",+0.034%
2020,163,204,794,"10,622,116","10,630,968",+0.083%
2021,189,211,735,"10,842,540","10,845,134",+0.024%
2022,51,229,904,"10,831,432","10,829,930",-0.014%
2023,41,209,681,"10,795,332","10,798,636",+0.031%


C2024 dictionary says: File Documentation for the Completions Data File, 2023-24 | (Provisional release)


In [5]:
c = pd.concat(comp.values(), ignore_index=True)
print(f"IPEDS completions, all US colleges, 2015-2024: {len(c):,} rows")

hd_cols = ["UNITID", "INSTNM", "CITY", "STABBR", "COUNTYCD", "SECTOR", "ICLEVEL", "CONTROL", "F1SYSNAM", "F1SYSCOD", "DEGGRANT", "CLOSEDAT"]
hd = pd.concat([read_member(RAW / "graduates/ipeds_directory" / f"HD{y}.zip",
                            next(m for m in zipfile.ZipFile(RAW / "graduates/ipeds_directory" / f"HD{y}.zip").namelist() if m.lower().endswith(".csv")),
                            usecols=hd_cols)[0].assign(year=y) for y in YEARS], ignore_index=True)
padded = {col: (hd[col] != hd[col].str.strip()).sum() for col in hd_cols}
print("HD cells with stray spaces:", {k: v for k, v in padded.items() if v})
# 2023+ files are space-padded; strip every text column (pandas 3 reads dtype=str as a string dtype, not object)
hd[hd_cols] = hd[hd_cols].apply(lambda s: s.str.strip())

cp = pd.read_csv(RAW / FILES[FILES.key == "cuny_programs"].file.iloc[0], dtype=str, keep_default_na=False)
print(f"\nCUNY Programs: {cp.shape[0]:,} rows x {cp.shape[1]} cols | columns: {list(cp.columns)}")

IPEDS completions, all US colleges, 2015-2024: 2,990,729 rows


HD cells with stray spaces: {'INSTNM': np.int64(1), 'F1SYSNAM': np.int64(20743), 'CLOSEDAT': np.int64(12090)}



CUNY Programs: 107,676 rows x 15 cols | columns: ['IRP CODE', 'COLLEGE', 'PROGRAM NAME', 'AWARD NAME', 'AWARD NUMBER', 'DATE FIRST REGISTERED', 'DATE CLOSED', 'TAP ELIGIBLE', 'APTS ELIGIBLE', 'VVTA ELIGIBLE', 'CIP 2020 CODE', 'CIP 2020 TITLE SHORT', 'RECORD TYPE', 'PERIOD', 'COUNT']


**Load findings**
- **Two encodings.** HD files are Latin-1 through 2022 and UTF-8 from 2023 (college names with accents break a UTF-8-only reader). Completions files are UTF-8 throughout.
- **Original vs revised.** 2015–2023 zips each hold the original *and* the revised file. The revision is small (about +0.02% to +0.08% of US awards: a few hundred rows corrected, added or dropped), but the revised file is the one to use. File names also change case (`c2015_a_rv.csv` → `C2023_a_RV.csv`), so match on `_rv` case-insensitively.
- **C2024 is provisional.** Its zip has no revised file, and the dictionary says *"Provisional release"*. Its numbers can still move by a fraction of a percent when the revised file comes out.
- HD text is space-padded from 2023 (20,743 `F1SYSNAM` cells, e.g. `"City University of New York"` + trailing spaces). A plain `==` test finds no CUNY colleges in 2023–2024. Strip every text column first. Watch out: with pandas 3, `dtype=str` gives a *string* dtype, not `object`, so a strip guarded by `dtype == object` silently does nothing.
- CUNY is identified by system code `F1SYSCOD = 136010`, which doesn't depend on how the name is spelled or padded.

## 2. What one row is (grain)

**Presumed keys:** completions `year × UNITID × CIPCODE × MAJORNUM × AWLEVEL`; directory `year × UNITID`; CUNY Programs `IRP CODE × RECORD TYPE × PERIOD`.

In [6]:
print("IPEDS completions duplicate keys:", c.duplicated(["year"] + KEY).sum())
print("IPEDS directory duplicate (year, UNITID):", hd.duplicated(["year", "UNITID"]).sum())

for k in [["IRP CODE"], ["IRP CODE", "COLLEGE"], ["IRP CODE", "AWARD NAME"], ["IRP CODE", "COLLEGE", "AWARD NAME"]]:
    print(f"CUNY Programs duplicates on {k + ['RECORD TYPE', 'PERIOD']}: {cp.duplicated(k + ['RECORD TYPE', 'PERIOD']).sum():,}")
progs = cp.drop_duplicates(["IRP CODE", "COLLEGE", "AWARD NAME"])
print(f"Distinct programs (IRP x college x award): {len(progs):,} | periods per program: {cp.groupby(['IRP CODE', 'COLLEGE', 'AWARD NAME']).size().unique()}")
progs[progs["IRP CODE"].duplicated(keep=False)].sort_values("IRP CODE")[["IRP CODE", "COLLEGE", "PROGRAM NAME", "AWARD NAME", "AWARD NUMBER"]].head(6)

IPEDS completions duplicate keys: 0
IPEDS directory duplicate (year, UNITID): 0
CUNY Programs duplicates on ['IRP CODE', 'RECORD TYPE', 'PERIOD']: 7,128
CUNY Programs duplicates on ['IRP CODE', 'COLLEGE', 'RECORD TYPE', 'PERIOD']: 4,968
CUNY Programs duplicates on ['IRP CODE', 'AWARD NAME', 'RECORD TYPE', 'PERIOD']: 792
CUNY Programs duplicates on ['IRP CODE', 'COLLEGE', 'AWARD NAME', 'RECORD TYPE', 'PERIOD']: 0


Distinct programs (IRP x college x award): 2,991 | periods per program: [36]


,IRP CODE,COLLEGE,PROGRAM NAME,AWARD NAME,AWARD NUMBER
51120,00140,John Jay College,Criminal Justice,BA,1 of 2
51156,00140,John Jay College,Criminal Justice,MA,2 of 2
6841,00156,Bronx Community College,Education,AAS,1 of 1
6840,00156,Bronx Community College,Education,AS,1 of 1
7057,00380,Bronx Community College,Accounting,AS,1 of 1
7056,00380,Bronx Community College,Accounting,AAS,1 of 1


**Grain findings**
- **IPEDS:** the key is unique in every year . One row = one college × one major × first/second major × one award level, for one award year.
- CUNY Programs: `IRP CODE` alone is not a key. One NYSED registration can carry several awards (Bronx CC Accounting = `AS` *and* `AAS` under `00380`; John Jay `00140` = a joint BA/MA, `AWARD NUMBER` "1 of 2" / "2 of 2"). The unique key is `IRP CODE × COLLEGE × AWARD NAME`: 2,991 programs, each with exactly 36 rows (18 academic years of graduates + 18 fall terms of enrollment). The file is a complete grid, so programs that are closed, or hadn't opened yet, carry zeros.

## 3. Column profile

In [7]:
lv = c.groupby("year").AWLEVEL.agg(lambda s: " ".join(sorted(s.unique(), key=lambda v: int(v))))
fl = c.groupby("year").XCTOTALT.agg(lambda s: dict(s.value_counts()))
mj = c.groupby("year").MAJORNUM.agg(lambda s: dict(s.value_counts()))
print("Non-numeric CTOTALT:", (~c.CTOTALT.str.fullmatch(r"\d+")).sum())
pd.DataFrame({"AWLEVEL values": lv, "XCTOTALT flags": fl, "MAJORNUM": mj})

Non-numeric CTOTALT: 0


,AWLEVEL values,XCTOTALT flags,MAJORNUM
year,,,
2015,01 02 03 04 05 06 07 08 17 18 19,"{'R': 300245, 'C': 13, 'N': 5}","{'1': 259726, '2': 40537}"
2016,01 02 03 04 05 06 07 08 17 18 19,"{'R': 302001, 'N': 27, 'C': 15}","{'1': 260470, '2': 41573}"
2017,01 02 03 04 05 06 07 08 17 18 19,"{'R': 308862, 'N': 16, 'C': 4}","{'1': 264694, '2': 44188}"
2018,01 02 03 04 05 06 07 08 17 18 19,"{'R': 289898, 'Z': 248, 'C': 3}","{'1': 268312, '2': 21837}"
2019,01 02 03 04 05 06 07 08 17 18 19,"{'R': 290962, 'P': 10}","{'1': 269505, '2': 21467}"
2020,02 03 04 05 06 07 08 17 18 19 20 21,"{'R': 287927, 'Z': 1862, 'P': 40, 'C': 4}","{'1': 268121, '2': 21712}"
2021,02 03 04 05 06 07 08 17 18 19 20 21,"{'R': 296320, 'P': 45}","{'1': 274922, '2': 21443}"
2022,02 03 04 05 06 07 08 17 18 19 20 21,"{'R': 301045, 'C': 10}","{'1': 279579, '2': 21476}"
2023,2 3 4 5 6 7 8 17 18 19 20 21,"{'R': 303458, 'C': 2}","{'1': 282430, '2': 21030}"


**IPEDS profile findings.** The dictionary (`C2024_A_Dict.zip`) defines the codes:
- **`AWLEVEL` changes twice.** (1) Values are zero-padded (`03`, `05`) through 2022 and unpadded (`3`, `5`) from 2023. Compare as integers. (2) From 2020, the old "< 1 year certificate" (`1`) is split into `20` (< 12 weeks) and `21` (12 weeks to 1 year). Associate's = `3` and bachelor's = `5` in every year.
- **`MAJORNUM`:** `1` = first major, `2` = second major. A double major is one award reported twice, so counting awards means `MAJORNUM = 1` only. Counting interest in a field can include second majors.
- **`XCTOTALT` imputation flag:** almost everything is `R` (reported). The rest are `P` (carried forward from last year), `C` (analyst corrected), `N` (nearest-neighbour imputed) or `Z` (implied zero). These are a few dozen rows a year, except 2020: 1,862 `Z` rows. Keep the flag.

In [8]:
print("CUNY Programs COUNT non-numeric:", (~cp.COUNT.str.fullmatch(r"\d+")).sum())
print("Rows with COUNT = 0:", f"{(cp.COUNT == '0').mean():.1%}")
for col in ["RECORD TYPE", "AWARD NUMBER", "TAP ELIGIBLE", "DATE CLOSED"]:
    vc = cp.drop_duplicates(["IRP CODE", "COLLEGE", "AWARD NAME"])[col].value_counts()
    print(f"{col:<14} (per program): {dict(vc.head(6))}{' …' if len(vc) > 6 else ''}")
print("\nAward names:", sorted(cp["AWARD NAME"].unique()))

CUNY Programs COUNT non-numeric: 0
Rows with COUNT = 0: 44.5%
RECORD TYPE    (per program): {'Enrollment': np.int64(2991)}
AWARD NUMBER   (per program): {'1 of 1': np.int64(2314), '1 of 2': np.int64(510), '2 of 2': np.int64(128), '1 of 3': np.int64(6), '1 of 4': np.int64(4), '2 of 18': np.int64(2)} …
TAP ELIGIBLE   (per program): {'YES': np.int64(2564), 'NO': np.int64(427)}
DATE CLOSED    (per program): {'': np.int64(2515), '10/19/2011': np.int64(25), '12/31/2018': np.int64(24), '08/19/2010': np.int64(24), '09/23/2011': np.int64(22), '07/02/2012': np.int64(20)} …

Award names: ['AA', 'AAS', 'ADPM', 'ADVCERPB', 'ADVCERPM', 'ADVCERT', 'AS', 'AUD', 'BA', 'BARCH', 'BBA', 'BE', 'BFA', 'BMUS', 'BPS', 'BS', 'BSED', 'BSNURS', 'BSW', 'BTECH', 'CERTGE30', 'CERTGE60', 'CERTLT30', 'DBA', 'DMA', 'DNP', 'DNS', 'DPH', 'DPT', 'DSW', 'EDD', 'JD', 'MA', 'MARCH', 'MAT', 'MBA', 'MD', 'ME', 'MFA', 'MIA', 'MLA', 'MLS', 'MM', 'MPA', 'MPH', 'MPS', 'MS', 'MSED', 'MSW', 'MUP', 'MUSB', 'NGASSOC', 'NGBACC', 'NGDO

**CUNY Programs profile findings**
- Counts are whole numbers with no suppression. 44.5% of rows are zero because of the full grid (§2).
- **55 award names**, CUNY's own short codes (`AA`, `AAS`, `BTECH`, `CERTGE30`, `NGBACC` …). IPEDS uses 12 award levels, so staging needs an award name → IPEDS level map (§4b).
- `DATE CLOSED` is filled for closed programs. A closed program keeps its history, so trend lines stay right.

## 4. Codes, hierarchy and special values

**(a) Which CIP edition?** CIP 2020 added 460 detailed codes and deleted or moved 152 old ones. Count both kinds in each year's file:

In [9]:
cip20 = pd.read_csv(RAW / "graduates/nces_cip/CIPCode2020.csv", dtype=str, keep_default_na=False)
cip20["CIPCode"] = cip20.CIPCode.str.strip('="')
xw1020 = pd.read_csv(RAW / "graduates/nces_cip/Crosswalk2010to2020.csv", dtype=str, keep_default_na=False).apply(lambda s: s.str.strip('="'))
detailed = lambda s: s[s.str.fullmatch(r"\d\d\.\d{4}")]
new_2020 = set(detailed(cip20[cip20.Action == "New"].CIPCode))
gone_2010 = set(detailed(xw1020[xw1020.Action.isin(["Deleted", "Moved to"])].CIPCode2010))
print("CIP codes in the IPEDS files: patterns", c.CIPCODE.str.replace(r"\d", "9", regex=True).value_counts().to_dict())
ed = []
for y, g in c.groupby("year"):
    codes = set(detailed(g.CIPCODE))
    ed.append({"year": y, "detailed CIP codes used": len(codes), "new-in-2020 codes": len(codes & new_2020),
               "deleted/moved-in-2020 codes": len(codes & gone_2010)})
pd.DataFrame(ed).set_index("year")

CIP codes in the IPEDS files: patterns {'99.9999': 2784191, '99': 206538}


,detailed CIP codes used,new-in-2020 codes,deleted/moved-in-2020 codes
year,,,
2015,1421,0,26
2016,1418,0,26
2017,1423,0,26
2018,1424,0,26
2019,1423,0,26
2020,1558,141,0
2021,1575,159,0
2022,1586,170,0
2023,1592,175,0


**CIP findings**
- 2015–2019 = CIP 2010; 2020–2024 = CIP 2020. The break is clean: before 2020 no new-2020 code appears and 26 retired codes do; from 2020 it's the reverse. The brief window starts at 2019 (AY 2018-19), so its first year is on the old edition and needs the CIP 2010 → 2020 crosswalk (notebook 06). The IPEDS dictionary confirms it: `CIPCODE` = *"CIP Code - 2020 Classification"*.
- Only two code shapes: `NN.NNNN` (detailed) and `99` (grand total per college × level × major number). There are no 2- or 4-digit subtotal rows, so detailed rows can be summed safely once `99` is excluded.

(b) CUNY award names → IPEDS award levels. CUNY's file has no level column, so build the map from the award names, then check it in §7:

In [10]:
AWARD_LEVEL = {  # CUNY award name -> IPEDS AWLEVEL group (from the award titles; checked against IPEDS in §7)
    **dict.fromkeys(["AA", "AS", "AAS", "NGASSOC"], "associate"),
    **dict.fromkeys(["BA", "BS", "BBA", "BFA", "BE", "BTECH", "BMUS", "BSED", "BPS", "BARCH", "BSNURS", "BSW", "MUSB", "NGBACC", "BID"], "bachelor"),
    **dict.fromkeys(["CERTLT30", "CERTGE30", "CERTGE60"], "certificate"),
}
cp["level"] = cp["AWARD NAME"].map(AWARD_LEVEL).fillna("graduate")
print("Mapped to graduate by default:", sorted(cp.loc[cp.level == "graduate", "AWARD NAME"].unique()))
display(cp.drop_duplicates(["IRP CODE", "COLLEGE", "AWARD NAME"]).groupby("level").size().rename("programs").to_frame().T)

print("CIP codes in CUNY Programs:", cp["CIP 2020 CODE"].str.replace(r"\d", "9", regex=True).value_counts().to_dict(),
      "| all in the CIP 2020 list:", cp["CIP 2020 CODE"].isin(set(cip20.CIPCode)).all())

Mapped to graduate by default: ['ADPM', 'ADVCERPB', 'ADVCERPM', 'ADVCERT', 'AUD', 'DBA', 'DMA', 'DNP', 'DNS', 'DPH', 'DPT', 'DSW', 'EDD', 'JD', 'MA', 'MARCH', 'MAT', 'MBA', 'MD', 'ME', 'MFA', 'MIA', 'MLA', 'MLS', 'MM', 'MPA', 'MPH', 'MPS', 'MS', 'MSED', 'MSW', 'MUP', 'NGDOC', 'PHD']


level,associate,bachelor,certificate,graduate
programs,382,1369,107,1133


CIP codes in CUNY Programs: {'99.9999': 107676} | all in the CIP 2020 list: True


**Award findings**
- `NGASSOC` and `NGBACC` are CUNY's non-departmental degrees (e.g. *C.U.N.Y. Baccalaureate (B.A.)*, the individualized degree). They're real associate's and bachelor's awards, so they map to those levels.
- `CERTLT30` / `CERTGE30` / `CERTGE60` are undergraduate certificates (under 30 credits, 30+, 60+). Out of v1 scope (associate + bachelor's), kept in staging.
- Every CUNY CIP code is a 6-digit CIP 2020 code, even for 2008 graduates: CUNY recoded its history to the current edition. That's convenient, but it means CUNY and IPEDS 2015–2019 disagree on codes for the same graduates.

## 5. Coverage

(a) Which colleges are in NYC? Use each year's directory: county code in the five boroughs. CUNY = system code `F1SYSCOD` `136010` (*City University of New York*).

In [11]:
hd["in_nyc"] = hd.COUNTYCD.isin(NYC_COUNTIES)
# CUNY's IPEDS system code; steadier than matching the system name text
hd["is_cuny"] = hd.F1SYSCOD == "136010"
SECTOR = {"0": "Administrative unit", "1": "Public 4-year+", "2": "Private nonprofit 4-year+", "3": "Private for-profit 4-year+",
          "4": "Public 2-year", "5": "Private nonprofit 2-year", "6": "Private for-profit 2-year",
          "7": "Public < 2-year", "8": "Private nonprofit < 2-year", "9": "Private for-profit < 2-year"}
nyc_hd = hd[hd.in_nyc]
display(nyc_hd.assign(sector=nyc_hd.SECTOR.map(SECTOR)).pivot_table(index="sector", columns="year", values="UNITID", aggfunc="size", fill_value=0))
print("CUNY units per year:", nyc_hd[nyc_hd.is_cuny].groupby("year").size().to_dict(),
      "| CUNY units outside NYC:", (hd.is_cuny & ~hd.in_nyc).sum())
print("Rows where CITY is a borough name but the county isn't NYC:",
      (hd.STABBR.eq("NY") & hd.CITY.isin(["New York", "Brooklyn", "Bronx", "Staten Island", "Flushing", "Jamaica"]) & ~hd.in_nyc).sum())

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
sector,,,,,,,,,,
Administrative unit,1,1,1,1,1,1,1,1,1,1
Private for-profit 2-year,18,18,16,13,14,16,15,16,17,16
Private for-profit 4-year+,11,11,10,10,10,10,9,9,8,8
Private for-profit < 2-year,40,42,40,39,36,36,34,31,28,25
Private nonprofit 2-year,14,12,13,11,12,13,13,13,13,14
Private nonprofit 4-year+,76,76,77,76,82,81,82,81,82,80
Private nonprofit < 2-year,7,7,6,6,5,4,4,4,4,5
Public 2-year,7,7,7,7,7,7,7,7,7,7
Public 4-year+,17,17,17,17,17,17,17,17,17,17


CUNY units per year: {2015: 21, 2016: 21, 2017: 21, 2018: 21, 2019: 21, 2020: 21, 2021: 21, 2022: 21, 2023: 21, 2024: 21} | CUNY units outside NYC: 0
Rows where CITY is a borough name but the county isn't NYC: 0


(b) How many associate's and bachelor's degrees do NYC colleges award? First majors only, from the `99` grand-total rows:

In [12]:
c["level"] = c.AWLEVEL.astype(int).map({3: "associate", 5: "bachelor"})
tot = c[(c.CIPCODE == "99") & (c.MAJORNUM == "1") & c.level.notna()].merge(hd[["year", "UNITID", "INSTNM", "in_nyc", "is_cuny"]], on=["year", "UNITID"], how="left")
print("Completions rows with no directory match (same year):", tot.in_nyc.isna().sum())
tot = tot[tot.in_nyc == True].assign(n=lambda d: pd.to_numeric(d.CTOTALT))
supply = tot.pivot_table(index="year", columns=["level", "is_cuny"], values="n", aggfunc="sum")
supply.columns = [f"{lvl}, {'CUNY' if cu else 'other NYC'}" for lvl, cu in supply.columns]
for lvl in ["associate", "bachelor"]:
    supply[f"{lvl}, CUNY share"] = supply[f"{lvl}, CUNY"] / (supply[f"{lvl}, CUNY"] + supply[f"{lvl}, other NYC"])
display(supply.style.format({c_: ("{:.0%}" if "share" in c_ else "{:,.0f}") for c_ in supply.columns}))

Completions rows with no directory match (same year): 0


,"associate, other NYC","associate, CUNY","bachelor, other NYC","bachelor, CUNY","associate, CUNY share","bachelor, CUNY share"
year,,,,,,
2015,"10,734","15,478","29,886","22,736",59%,43%
2016,"10,107","16,345","30,475","23,709",62%,44%
2017,"9,139","17,372","29,799","24,456",66%,45%
2018,"8,791","18,247","29,785","25,371",67%,46%
2019,"8,713","18,880","30,509","27,273",68%,47%
2020,"8,319","18,498","32,114","27,219",69%,46%
2021,"8,349","17,596","31,295","29,590",68%,49%
2022,"7,554","14,662","31,749","28,008",66%,47%
2023,"6,717","12,246","30,949","25,074",65%,45%


In [13]:
# Who awards the non-CUNY associate's degrees?
a24 = tot[(tot.year == 2024) & (tot.level == "associate") & ~tot.is_cuny.astype(bool)].nlargest(8, "n")[["INSTNM", "n"]]
a24

,INSTNM,n
50075,Fashion Institute of Technology,1914
50135,Monroe University,942
50279,Yeshiva University,514
50024,St Paul's School of Nursing-Queens,298
50252,Touro University,296
50164,Plaza College,263
50204,St Paul's School of Nursing-Staten Island,239
50245,Swedish Institute a College of Health Sciences,210


**Coverage findings**
- NYC has 191 units in 2015 and 173 in 2024. The largest group is private nonprofit 4-year (76–82 units). The fall is almost all small for-profit schools closing (for-profit < 2-year: 40 → 25). A college that closed still has its past completions in the earlier files, so use each year's own directory, not the latest one, to decide what was in NYC.
- CUNY = 21 units in every year, including the System Office (awards nothing) and two graduate-only schools (Graduate Center, Law). None is outside NYC.
- CUNY awards most of NYC's associate's degrees: 59% to 69% depending on the year (62% in 2024), so the associate's side of this project is largely a CUNY story. For the bachelor's, CUNY's share is 43% to 49%, with NYU, Columbia, St. John's, Pace and the rest making up the majority.
- **The associate's collapse.** NYC associate's degrees fell by a third between 2019 and 2024 (27,593 → 18,490), and about four-fifths of the drop is CUNY (18,880 → 11,402). CUNY bachelor's degrees also fell from their 2021 peak. That's a headline finding for Q3, and it matches CUNY's own published counts (§7), and CUNY Programs shows it flattening in 2025 (11,676 associate's).
- Every completions row finds its college in the same year's directory.

In [14]:
print("CUNY Programs periods:", cp.groupby("RECORD TYPE").PERIOD.agg(lambda s: f"{s.min()} … {s.max()} ({s.nunique()})").to_dict())
print("CUNY Programs colleges:", sorted(cp.COLLEGE.unique()))

CUNY Programs periods: {'Enrollment': 'Fall_2008 … Fall_2025 (18)', 'Graduation': 'AY_2007_2008 … AY_2024_2025 (18)'}
CUNY Programs colleges: ['Baruch College', 'Borough of Manhattan Community College', 'Bronx Community College', 'Brooklyn College', 'CUNY Law School', 'City College', 'Graduate Center', 'Guttman Community College', 'Hostos Community College', 'Hunter College', 'John Jay College', 'Kingsborough Community College', 'LaGuardia Community College', 'Labor and Urban Studies', 'Lehman College', 'Medgar Evers College', 'Medical School (CC)', 'Medical School (GC)', 'NYC College of Technology', 'Queens College', 'Queensborough Community College', 'School Prof Studies (GC)', 'School of Journalism', 'School of Public Health', 'The College of Staten Island', 'York College']


**CUNY Programs coverage:** graduates for AY 2007-08 → 2024-25 and enrollment for Fall 2008 → Fall 2025. That's one more year of graduates than IPEDS has (C2024 = AY 2023-24), and enrollment, which IPEDS completions doesn't have at all (enrollment = the pipeline of future graduates). The college list has more units than IPEDS (e.g. *School of Journalism*, *School of Public Health*, *Labor and Urban Studies*, *Medical School (CC/GC)*). IPEDS reports these under a parent college.

## 6. Relationships: will it join?

Three joins: (1) completions → directory on `year × UNITID` (in §5); (2) CUNY Programs college → IPEDS `UNITID` (names differ, so a hand map); (3) majors → jobs, CIP 2020 → SOC 2018 (built in notebook 06, but measure the match rate now).

In [15]:
cuny_ids = hd[(hd.year == 2024) & hd.is_cuny].set_index("INSTNM").UNITID
COLLEGE_TO_UNITID = {  # CUNY Programs COLLEGE -> IPEDS UNITID (parent college for units IPEDS doesn't list)
    "Baruch College": "190512", "Borough of Manhattan Community College": "190521", "Bronx Community College": "190530",
    "Brooklyn College": "190549", "The College of Staten Island": "190558", "City College": "190567",
    "Graduate Center": "190576", "Hostos Community College": "190585", "Hunter College": "190594",
    "John Jay College": "190600", "Kingsborough Community College": "190619", "LaGuardia Community College": "190628",
    "Lehman College": "190637", "Medgar Evers College": "190646", "NYC College of Technology": "190655",
    "Queens College": "190664", "Queensborough Community College": "190673", "CUNY Law School": "190682",
    "York College": "190691", "Guttman Community College": "475565",
    "Medical School (CC)": "190567", "Medical School (GC)": "190576", "School Prof Studies (GC)": "190576",
    "School of Journalism": "190576", "School of Public Health": "190576", "Labor and Urban Studies": "190576",
}
unmapped = sorted(set(cp.COLLEGE) - set(COLLEGE_TO_UNITID))
print("CUNY Programs colleges without a UNITID:", unmapped or "none")
print("Mapped UNITIDs that aren't CUNY in HD 2024:", sorted(set(COLLEGE_TO_UNITID.values()) - set(cuny_ids)) or "none")

CUNY Programs colleges without a UNITID: none
Mapped UNITIDs that aren't CUNY in HD 2024: none


In [16]:
cs = pd.read_excel(RAW / "graduates/nces_cip/CIP2020_SOC2018_Crosswalk.xlsx", sheet_name="CIP-SOC", dtype=str)
has_job = set(cs[~cs.SOC2018Code.str.startswith("99")].CIP2020Code)
d24 = c[(c.year == 2024) & (c.MAJORNUM == "1") & c.level.notna() & (c.CIPCODE != "99")].merge(
    hd[hd.year == 2024][["UNITID", "in_nyc"]], on="UNITID")
d24 = d24[d24.in_nyc].assign(n=lambda d: pd.to_numeric(d.CTOTALT), linked=lambda d: d.CIPCODE.isin(has_job))
print(f"NYC associate + bachelor's awards, 2024: {d24.n.sum():,} | with at least one job in the CIP→SOC crosswalk: {d24.n[d24.linked].sum() / d24.n.sum():.1%}")
print("Largest majors with no linked job:")
d24[~d24.linked].groupby("CIPCODE").n.sum().nlargest(6).rename("awards").to_frame().join(cip20.set_index("CIPCode").CIPTitle)

NYC associate + bachelor's awards, 2024: 73,225 | with at least one job in the CIP→SOC crosswalk: 97.1%
Largest majors with no linked job:


,awards,CIPTitle
CIPCODE,,
51.0000,1020,"Health Services/Allied Health/Health Sciences, General."
50.0499,199,"Design and Applied Arts, Other."
52.9999,178,"Business, Management, Marketing, and Related Support Services, Other."
30.0000,140,"Multi-/Interdisciplinary Studies, General."
30.9999,94,"Multi-/Interdisciplinary Studies, Other."
11.0899,82,"Computer Software and Media Applications, Other."


**Join findings**
- **CUNY Programs → IPEDS:** all 26 CUNY Programs units map to one of the 20 award-granting CUNY UNITIDs. Six units (journalism, public health, labor studies, SPS, the two medical-school entries) have no IPEDS record of their own, so they are mapped to their parent (Graduate Center or City College). That's an assumption, checked by the college-level tie-out in §7.
- **Majors → jobs:** about 97% of NYC associate + bachelor's awards are in a CIP code that the official crosswalk links to at least one job. The gap is mostly general or "other" majors (`51.0000` Health Services/Allied Health, General; `30.0000` Multi/Interdisciplinary Studies; `…99` "Other" codes). The crosswalk deliberately gives these `99-9999` (no job). Notebook 06 decides how to treat them; they are visible in the dashboard as "no direct job link".

## 7. Tie-out

(a) IPEDS vs CUNY's own published count. CUNY's *Degrees Granted by Type* (`ybg5-afvs`) gives associate's and bachelor's per year. Compare with IPEDS CUNY totals (first majors):

In [17]:
dt = pd.read_csv(RAW / FILES[FILES.key == "cuny_degrees_by_type"].file.iloc[0], dtype=str, keep_default_na=False)
print("Degree Type labels:", sorted(dt["Degree Type"].unique()))
print("Record Type labels:", sorted(dt["Record Type"].unique()))
dt["type"] = dt["Degree Type"].str.upper().replace({"BACCALAUREATE": "BACHELOR'S"})
dt["year"] = dt["Academic Year"].str[-4:].astype(int)
pub = dt[dt.type.isin(["ASSOCIATE", "BACHELOR'S"])].pivot_table(index="year", columns="type", values="Count", aggfunc="first").astype(int)
ipeds_cuny = tot[tot.is_cuny.astype(bool)].pivot_table(index="year", columns="level", values="n", aggfunc="sum")
check = pd.DataFrame({"CUNY published, associate": pub["ASSOCIATE"], "IPEDS, associate": ipeds_cuny["associate"],
                      "CUNY published, bachelor's": pub["BACHELOR'S"], "IPEDS, bachelor's": ipeds_cuny["bachelor"]}).loc[2019:2024]
check["match"] = (check.iloc[:, 0] == check.iloc[:, 1]) & (check.iloc[:, 2] == check.iloc[:, 3])
display(check)
print("Duplicate (year, type) rows in the published file:", dt.duplicated(["year", "type"]).sum())

Degree Type labels: ['ADVANCED CERTIFICATE', 'ASSOCIATE', 'Advance Certificate', 'Associate', "BACHELOR'S", 'Baccalaureate', 'CERTIFICATE', 'Certificate', 'DOCTORAL', 'FIRST-PROFESSIONAL', 'J.D.', 'JD', "MASTER'S", "MASTER'S OF PHILOSOPHY", 'MD', 'Master of Philosophy', "Master's", 'PhD', 'Sum', 'TOTAL', 'Total']
Record Type labels: ['DEGREE', 'DEGREES', 'Degrees']


,"CUNY published, associate","IPEDS, associate","CUNY published, bachelor's","IPEDS, bachelor's",match
year,,,,,
2019,18880,18880.0,27273,27273.0,True
2020,18499,18498.0,27219,27219.0,False
2021,17596,17596.0,29590,29590.0,True
2022,14662,14662.0,28008,28008.0,True
2023,12246,12246.0,25074,25074.0,True
2024,11402,11402.0,23437,23437.0,True


Duplicate (year, type) rows in the published file: 0


**(b) CUNY Programs vs IPEDS.** Sum CUNY Programs graduates by level and compare with IPEDS CUNY, first majors *and* first + second majors:

In [18]:
g = cp[cp["RECORD TYPE"] == "Graduation"].assign(n=lambda d: pd.to_numeric(d.COUNT),
                                                 year=lambda d: d.PERIOD.str[-4:].astype(int))
prog_tot = g[g.level.isin(["associate", "bachelor"])].pivot_table(index="year", columns="level", values="n", aggfunc="sum")
both_majors = c[(c.CIPCODE == "99") & c.level.notna() & c.UNITID.isin(set(cuny_ids))].assign(
    n=lambda d: pd.to_numeric(d.CTOTALT)).pivot_table(index="year", columns="level", values="n", aggfunc="sum")
cmp = pd.DataFrame({
    "CUNY Programs, associate": prog_tot["associate"], "IPEDS 1st major, associate": ipeds_cuny["associate"],
    "CUNY Programs, bachelor's": prog_tot["bachelor"], "IPEDS 1st major, bachelor's": ipeds_cuny["bachelor"],
    "IPEDS 1st+2nd major, bachelor's": both_majors["bachelor"]}).loc[2019:2025]
cmp.style.format("{:,.0f}", na_rep="—")

,"CUNY Programs, associate","IPEDS 1st major, associate","CUNY Programs, bachelor's","IPEDS 1st major, bachelor's","IPEDS 1st+2nd major, bachelor's"
year,,,,,
2019,"18,881","18,880","28,117","27,273","27,983"
2020,"18,498","18,498","28,023","27,219","27,908"
2021,"17,548","17,596","30,244","29,590","29,659"
2022,"14,636","14,662","28,557","28,008","28,665"
2023,"12,188","12,246","25,788","25,074","25,849"
2024,"11,364","11,402","24,085","23,437","24,154"
2025,"11,676",—,"22,591",—,—


In [19]:
# College by college, AY 2023-24, bachelor's: CUNY Programs vs IPEDS first + second majors
g24 = g[(g.year == 2024) & (g.level == "bachelor")].assign(UNITID=lambda d: d.COLLEGE.map(COLLEGE_TO_UNITID)).groupby("UNITID").n.sum()
i24 = c[(c.year == 2024) & (c.CIPCODE == "99") & (c.level == "bachelor")].assign(n=lambda d: pd.to_numeric(d.CTOTALT)).groupby("UNITID").n.sum()
by_col = pd.DataFrame({"college": cuny_ids.reset_index().set_index("UNITID").INSTNM, "CUNY Programs": g24, "IPEDS 1st+2nd": i24}).dropna(subset=["CUNY Programs"])
by_col["difference"] = by_col["CUNY Programs"] - by_col["IPEDS 1st+2nd"]
by_col.sort_values("difference")

,college,CUNY Programs,IPEDS 1st+2nd,difference
UNITID,,,,
190576,CUNY Graduate School and University Center,709.0,787.0,-78.0
190512,CUNY Bernard M Baruch College,3505.0,3506.0,-1.0
190600,CUNY John Jay College of Criminal Justice,2893.0,2894.0,-1.0
190567,CUNY City College,2033.0,2034.0,-1.0
190558,College of Staten Island CUNY,1344.0,1344.0,0.0
190549,CUNY Brooklyn College,2476.0,2476.0,0.0
190637,CUNY Lehman College,2388.0,2388.0,0.0
190646,CUNY Medgar Evers College,367.0,367.0,0.0
190691,CUNY York College,935.0,935.0,0.0


**(c) Internal arithmetic.** In IPEDS, each `99` grand-total row should equal the sum of its detailed CIP rows:

In [20]:
cn = c.assign(n=pd.to_numeric(c.CTOTALT))
gt = cn[cn.CIPCODE == "99"].set_index(["year", "UNITID", "MAJORNUM", "AWLEVEL"]).n
dsum = cn[cn.CIPCODE != "99"].groupby(["year", "UNITID", "MAJORNUM", "AWLEVEL"]).n.sum()
j = pd.concat([gt.rename("total"), dsum.rename("sum_detail")], axis=1).fillna(0)
print(f"Grand-total rows checked: {len(j):,} | equal to the sum of detail: {(j.total == j.sum_detail).mean():.2%}")
mw = cn.assign(m=pd.to_numeric(cn.CTOTALM), w=pd.to_numeric(cn.CTOTALW))
print(f"Men + women = total: {(mw.m + mw.w == mw.n).mean():.2%} of rows")

Grand-total rows checked: 206,538 | equal to the sum of detail: 100.00%


Men + women = total: 100.00% of rows


**Tie-out findings**
- (a) Exact in every year 2019–2024 except one award in 2020. CUNY's published associate's and bachelor's counts are identical to IPEDS CUNY first-major totals (e.g. 2024: 11,402 / 23,437). The one exception is 2020 associate's: CUNY 18,499 vs IPEDS 18,498 (±1, likely a late correction on one side). So the *Degrees Granted by Type* dataset is IPEDS, not an independent count. It confirms our CUNY filter and the first-major rule, and it confirms the provisional C2024 matches CUNY's own final figure. The file itself is messy: degree-type and record-type labels change case and wording across years (`Baccalaureate` / `BACHELOR'S`, `DEGREES` / `DEGREE`).
- (b) CUNY Programs is close to IPEDS, but counts differently. Associate's are within a few dozen of IPEDS. Bachelor's run 550–850 a year higher than IPEDS first majors and close to first + second majors (within 134, except 2021 at +585): CUNY Programs counts a degree under every program it completes, so a double major counts twice. For "graduates per major" that's the right count (it's what IPEDS second majors are for); for "how many people graduated" it overcounts. College by college (2024 bachelor's vs IPEDS 1st + 2nd majors) the gap is 0 or ±1 at ten of twelve colleges, which supports the parent-college mapping of the extra units. The two exceptions: Graduate Center −78 (CUNY Programs has fewer; not yet explained, to check in staging) and Hunter +11.
- **(c) ** Every IPEDS grand-total row equals the sum of its detailed rows, and men + women = total.

## 8. Fitness for purpose

| Question | Source | Fit | Why |
|---|---|---|---|
| Q3: graduates per major per year, all NYC colleges | IPEDS | Use | Complete census, every NYC college, by CIP and level, 2015–2024. Caveats: awards not people; CIP 2010 before 2020; C2024 provisional |
| **Q3: CUNY highlighted** | IPEDS (+ HD) | yes | `F1SYSCOD = 136010` flag. Ties to CUNY's published totals (exact except ±1 in 2020) |
| Q3: CUNY graduates for the latest year (2024-25) | CUNY Programs | Use for CUNY's latest year and for enrollment | One year ahead of IPEDS. Counts each program completed (double majors twice); CUNY-only, so it can't be the all-NYC number |
| Pipeline: CUNY enrollment by program | CUNY Programs | yes | Fall 2008–2025. No IPEDS equivalent at program level |
| Supply vs demand (graduates per opening) | IPEDS + CIP→SOC + projections | partial | Works for ~97% of awards; general/"other" majors have no direct job. One award ≠ one job seeker (graduates leave NYC, go to grad school, or already work) |
| Where graduates end up working | — | no | Not in these files. ACS PUMS (notebook 08) |
| Graduates by borough of residence | — | no | Only the college's location |

## 9. Issues list and proposed staging tables

| # | Issue | Source · rows affected | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | Original + revised CSV in the same zip; name case varies | IPEDS · 2015–2023 | Read the `_rv` member (case-insensitive); else the only member, `release = 'provisional'` |
| 2 | C2024 is provisional | IPEDS · 2024 | `release` column; re-download when the revised file is out (follow-up) |
| 3 | Latin-1 HD files through 2022 | IPEDS HD · 2015–2022 | Try UTF-8, fall back to Latin-1 |
| 4 | Padded `F1SYSNAM` (and other HD text) | IPEDS HD · 2023–2024 | Strip every text column; flag CUNY on `F1SYSCOD = 136010` |
| 5 | `AWLEVEL` zero-padding changes in 2023; level 1 split into 20/21 from 2020 | IPEDS · all | Cast to int; `award_level` dimension with both old and new certificate codes |
| 6 | Second majors double-count awards | IPEDS · ~7% of rows | Keep `major_number`; measures say whether they count awards (1st only) or majors (1st + 2nd) |
| 7 | `99` grand-total rows mixed with detail | IPEDS · ~20k rows/year | Drop from the fact table (they're derivable); keep the tie-out test |
| 8 | CIP 2010 in 2015–2019 | IPEDS · 5 years incl. brief start year 2019 | Keep `cip_code` + `cip_edition`; map to CIP 2020 in core (nb 06) |
| 9 | Imputation flags (`P`, `C`, `N`, `Z`) | IPEDS · dozens/year; 1,862 `Z` in 2020 | Keep as `imputation_flag` |
| 10 | Colleges close; NYC membership changes | IPEDS HD · 191 → 173 units | NYC flag from each year's HD (county code) |
| 11 | `IRP CODE` not unique (several awards per registration) | CUNY Programs · 365 programs under 167 IRP codes | Key `irp_code × college × award_name` |
| 12 | Full grid of zeros (closed / not yet open programs) | CUNY Programs · 44.5% of rows | Keep zeros (they're real for open programs); `date_closed` tells them apart |
| 13 | No award level; 55 CUNY award names | CUNY Programs · all | `AWARD_LEVEL` map (§4b) → `award_level`; assert no unmapped undergraduate names |
| 14 | Units with no IPEDS record (journalism, public health …) | CUNY Programs · 6 units | Hand map to parent UNITID (§6), flagged as assumption |
| 15 | Counts a degree once per program (double majors twice) | CUNY Programs · bachelor's | Document; use IPEDS for "how many graduates", CUNY Programs for the latest year and enrollment |
| 16 | History recoded to CIP 2020 | CUNY Programs · all | Note: CUNY 2015–19 CIP ≠ IPEDS 2015–19 CIP for the same graduates |
| 17 | Wide/long mix: graduates and enrollment in one column | CUNY Programs · all | Split into two staging tables on `RECORD TYPE` |
| 18 | Messy labels, duplicated label variants | CUNY degrees by type | Tie-out only; not staged |

**Proposed staging table 1:** `stg_ipeds__completions`
- **Grain / key:** `year × unitid × cip_code × major_number × award_level`, detailed CIP only, NYC colleges only (each year's HD). Key asserted unique.
- **Columns:** `year` int (award year end), `release` text, `unitid` text, `cip_code` text, `cip_edition` text (`2010`/`2020`), `major_number` smallint, `award_level` smallint, `awards_total`, `awards_men`, `awards_women` int, `imputation_flag` text, `source_file` text. Race/ethnicity columns stay in raw for now (not in v1 questions).

**Proposed staging table 2:** `stg_ipeds__institutions`
- **Grain / key:** `year × unitid`, NYC only. Columns: `year`, `unitid`, `name`, `city`, `county_fips`, `borough`, `sector`, `control`, `level`, `is_cuny`, `system_name`, `closed_date`.

**Proposed staging tables 3–4:** `stg_cuny__program_graduates`, `stg_cuny__program_enrollment`
- **Grain / key:** `irp_code × college × award_name × academic_year` (graduates) and `… × fall_term` (enrollment).
- **Columns:** `irp_code`, `college`, `unitid` (mapped), `program_name`, `award_name`, `award_level`, `award_number`, `cip_code` (2020), `date_registered` date, `date_closed` date, `tap_eligible` bool, `period` text, `year` int, `graduates` / `enrolled` int, `source_file`.

In [21]:
# Row accounting for the proposed staging filters, recomputed rather than typed
nyc_ids = hd[hd.in_nyc][["year", "UNITID"]]
stg = c[c.CIPCODE != "99"].merge(nyc_ids, on=["year", "UNITID"])
assert not stg.duplicated(["year"] + KEY).any()
print(f"IPEDS completions rows (US, all years): {len(c):,} -> NYC detailed rows: {len(stg):,}")
print("   per year:", stg.groupby("year").size().to_dict())
print(f"IPEDS institutions (NYC, all years): {len(nyc_hd):,}")
for rt, name in [("Graduation", "stg_cuny__program_graduates"), ("Enrollment", "stg_cuny__program_enrollment")]:
    s = cp[cp["RECORD TYPE"] == rt]
    assert not s.duplicated(["IRP CODE", "COLLEGE", "AWARD NAME", "PERIOD"]).any()
    print(f"{name}: {len(s):,} rows")

IPEDS completions rows (US, all years): 2,990,729 -> NYC detailed rows: 58,343
   per year: {2015: 5337, 2016: 5533, 2017: 5911, 2018: 5942, 2019: 5984, 2020: 5928, 2021: 5824, 2022: 6019, 2023: 5924, 2024: 5941}
IPEDS institutions (NYC, all years): 1,836
stg_cuny__program_graduates: 53,838 rows
stg_cuny__program_enrollment: 53,838 rows
